# 02. NNI 산출

뉴스 유목민 지수(NNI) 분석 파이프라인 2/3 단계.
01_전처리.ipynb의 산출물을 이어받아 H_norm(채널 다양성)·D_weighted(빈도가중 이탈도)를 계산하고,
PCA 기반 가중치로 NNI를 산출한 뒤 구성타당도를 검증한다.

- 입력: `step1_channels.pkl`
- 출력: `step2_nni.pkl` (다음 단계 03_군집분석.ipynb 에서 사용)

In [1]:
# =========================================================
# 0. 라이브러리 & 01단계 산출물 불러오기
# =========================================================
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pickle

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 240)

with open('step1_channels.pkl', 'rb') as f:
    _s1 = pickle.load(f)
df = _s1['df']
channel_specs = _s1['channel_specs']
channel_cols = _s1['channel_cols']
channel_tiers = _s1['channel_tiers']
FREQ_WINDOW = _s1['FREQ_WINDOW']
print('01단계 산출물 로드 완료. df shape:', df.shape)

01단계 산출물 로드 완료. df shape: (6000, 234)


## 4. 뉴스 유목민 지수(NNI) 계산

- **H_norm**: 채널 이용의 섀넌 엔트로피(분산도), 0~1 정규화
- **D_weighted**: 실제 이용한 모든 채널의 **빈도 가중 평균** 이탈도 (기존처럼 주 이용경로 1문항에만
  의존하지 않으므로 결측/무응답에 안전하고 정도의 차이도 반영됨)
- **뉴스 비활성**: 이번 주 뉴스 채널 이용이 전혀 없는 사람은 H/D 자체가 정의되지 않음.
  **표본에서 제외하지 않고** `news_inactive` 플래그로 남겨서, NNI/군집 계산 대상에서는 빠지되
  8절에서 별도 세그먼트로 그대로 분석·보고한다.

In [2]:
# =========================================================
# 4. H_norm / D_weighted / NNI
# =========================================================
def shannon_entropy_from_row(values):
    arr = np.asarray(values, dtype=float)
    total = arr.sum()
    if total <= 0:
        return np.nan
    p = arr[arr > 0] / total
    return -(p * np.log2(p)).sum()

H_max = np.log2(len(channel_cols))
df['H'] = df[channel_cols].apply(lambda row: shannon_entropy_from_row(row.values), axis=1)
df['H_norm'] = df['H'] / H_max

freq_matrix = df[channel_cols].to_numpy(dtype=float)
row_total   = freq_matrix.sum(axis=1)
with np.errstate(invalid='ignore', divide='ignore'):
    weighted_tier = (freq_matrix * channel_tiers).sum(axis=1) / row_total
df['D_weighted'] = np.where(row_total > 0, weighted_tier / 2.0, np.nan)  # tier 최댓값 2로 정규화

# 뉴스 비활성 플래그 (제외가 아니라 별도 분석 대상으로 유지 -> 8절)
df['news_inactive'] = row_total == 0
n_inactive = df['news_inactive'].sum()
print(f'뉴스 비활성 응답자: {n_inactive}명 ({round(n_inactive/len(df)*100,1)}%) — 8절에서 별도 분석')

# 데이터 기반 가중치(alpha, beta): H_norm/D_weighted 표준화 후 PCA 1주성분 적재값
valid = ~df['news_inactive']
hd = df.loc[valid, ['H_norm', 'D_weighted']].dropna()
hd_scaled = StandardScaler().fit_transform(hd)
pca_hd = PCA(n_components=2, random_state=42).fit(hd_scaled)
loadings = pca_hd.components_[0]
explained_ratio = pca_hd.explained_variance_ratio_[0]
if loadings.sum() < 0:
    loadings = -loadings
alpha, beta = loadings / loadings.sum()

corr_hd, _ = pearsonr(hd['H_norm'], hd['D_weighted'])
print(f'H_norm-D_weighted 상관 r={round(corr_hd,3)}, PC1 설명분산={round(explained_ratio*100,1)}%')
print(f'데이터 기반 가중치 -> alpha(H_norm)={round(alpha,3)}, beta(D_weighted)={round(beta,3)}')
# 검증: (a) WT 설계가중치를 반영해도 alpha/beta 거의 그대로(차이 0.0)
#       (b) alpha를 0~1 전 구간에서 바꿔도 실루엣 변동폭 0.0075에 불과(3-3b절 그리드서치)
#       -> 가중치는 실루엣의 원인이 아니었으므로 임의 조정 없이 위 PCA값을 그대로 채택.

df['NNI'] = np.where(df['news_inactive'], np.nan, alpha * df['H_norm'] + beta * df['D_weighted'])
df['NNI_percentile'] = df['NNI'].rank(pct=True) * 100

print()
print(df.loc[valid, ['H_norm', 'D_weighted', 'NNI', 'NNI_percentile']].describe().round(3))

뉴스 비활성 응답자: 298명 (5.0%) — 8절에서 별도 분석
H_norm-D_weighted 상관 r=0.658, PC1 설명분산=82.9%
데이터 기반 가중치 -> alpha(H_norm)=0.5, beta(D_weighted)=0.5

         H_norm  D_weighted       NNI  NNI_percentile
count  5702.000    5702.000  5702.000        5702.000
mean      0.284       0.304     0.294          50.009
std       0.236       0.264     0.227          28.606
min      -0.000       0.000     0.000          13.048
25%       0.000       0.000     0.000          13.048
50%       0.285       0.250     0.270          50.421
75%       0.455       0.500     0.479          74.649
max       0.983       1.000     0.844         100.000


In [3]:
# =========================================================
#  H_norm 구성타당도 검증 — 붙일 위치: 셀 8(H_norm/D_weighted/NNI 정의) 다음
#   지적: H_norm ~ total_freq r=.824 → "H가 다양성이 아니라 총이용량을 재는 것 아닌가?"
#   반박: 그 상관은 n_channels(이용 채널 수)를 경유한 허위상관.
#   ※ 이 노트북 변수(channel_cols, df['H_norm'], df['D_weighted'], df['news_inactive'])만 사용
# =========================================================
from scipy.stats import pearsonr
import statsmodels.api as sm

df['total_freq'] = df[channel_cols].sum(axis=1)          # 총 이용일수 (관여도 대리)
df['n_channels'] = (df[channel_cols] > 0).sum(axis=1)    # 이용 채널 수 (다양성 폭)

v = df.loc[~df['news_inactive']].copy()                  # H/D가 정의되는 활성 응답자
print(f'[검증 대상] 활성 응답자 n={len(v)} (뉴스 비활성 {int(df["news_inactive"].sum())}명 제외)\n')

[검증 대상] 활성 응답자 n=5702 (뉴스 비활성 298명 제외)



In [4]:
# ---------- (1) 단순 상관 ----------
print('[1] 단순 상관')
for a, b, note in [('H_norm', 'total_freq', '지적된 값'),
                   ('H_norm', 'n_channels', 'H가 실제로 재는 것'),
                   ('n_channels', 'total_freq', '공통원인 경로')]:
    print(f'  {a:11s} ~ {b:11s} r = {pearsonr(v[a], v[b])[0]:+.3f}   ({note})')

[1] 단순 상관
  H_norm      ~ total_freq  r = +0.824   (지적된 값)
  H_norm      ~ n_channels  r = +0.950   (H가 실제로 재는 것)
  n_channels  ~ total_freq  r = +0.842   (공통원인 경로)


In [5]:
# ---------- (2) 편상관 — 핵심 반박 ----------
def partial_corr(y, x, ctrl):
    ry = sm.OLS(v[y], sm.add_constant(v[ctrl])).fit().resid
    rx = sm.OLS(v[x], sm.add_constant(v[ctrl])).fit().resid
    return pearsonr(ry, rx)[0]

print('\n[2] 편상관 (핵심 반박)')
print(f"  H_norm ~ total_freq | n_channels 통제 : "
      f"{pearsonr(v.H_norm, v.total_freq)[0]:+.3f} -> {partial_corr('H_norm','total_freq',['n_channels']):+.3f}   (붕괴=허위상관)")
print(f"  H_norm ~ n_channels | total_freq 통제 : "
      f"{pearsonr(v.H_norm, v.n_channels)[0]:+.3f} -> {partial_corr('H_norm','n_channels',['total_freq']):+.3f}   (유지=진짜 신호)")
print('  => H_norm은 관여도(총량)가 아니라 채널 폭(richness)을 측정한다.')


[2] 편상관 (핵심 반박)
  H_norm ~ total_freq | n_channels 통제 : +0.824 -> +0.142   (붕괴=허위상관)
  H_norm ~ n_channels | total_freq 통제 : +0.950 -> +0.837   (유지=진짜 신호)
  => H_norm은 관여도(총량)가 아니라 채널 폭(richness)을 측정한다.


In [6]:
# ---------- (3) 총량 통제 시에도 H가 분산하는가 ----------
print('\n[3] 총 이용일수 구간별 H_norm 분포 (같은 총량 안에서도 H가 다르면 OK)')
v['tf_bin'] = pd.cut(v['total_freq'], [0, 7, 14, 21, 28, 100],
                     labels=['1-7일', '8-14일', '15-21일', '22-28일', '29일+'])
print(v.groupby('tf_bin', observed=True).agg(
    n=('H_norm', 'size'), H_mean=('H_norm', 'mean'), H_sd=('H_norm', 'std'),
    H_min=('H_norm', 'min'), H_max=('H_norm', 'max')).round(3).to_string())


[3] 총 이용일수 구간별 H_norm 분포 (같은 총량 안에서도 H가 다르면 OK)
           n  H_mean   H_sd  H_min  H_max
tf_bin                                   
1-7일    2360   0.063  0.126 -0.000  0.747
8-14일   1926   0.343  0.107  0.157  0.729
15-21일   863   0.517  0.089  0.372  0.848
22-28일   371   0.613  0.068  0.514  0.889
29일+     182   0.739  0.078  0.621  0.983


In [7]:
# ---------- (4) H_pure: 관여도 잔차화 (민감도 분석용, 셀 B에서 사용) ----------
print('\n[4] H_pure — total_freq 잔차화 버전')
_ols = sm.OLS(v['H_norm'], sm.add_constant(v['total_freq'])).fit()
_res = _ols.resid
v['H_pure'] = (_res - _res.min()) / (_res.max() - _res.min())   # 0~1 재정규화
df['H_pure'] = np.nan
df.loc[v.index, 'H_pure'] = v['H_pure'].values                 # 비활성자는 NaN (D_weighted와 동일 처리)
print(f'  R^2(total_freq -> H_norm) = {_ols.rsquared:.3f}')
print(f'  H_pure ~ total_freq r = {pearsonr(v.H_pure, v.total_freq)[0]:+.3f}  (설계상 0)')
print(f'  H_pure ~ H_norm     r = {pearsonr(v.H_pure, v.H_norm)[0]:+.3f}')
print(f'  H_pure ~ D_weighted r = {pearsonr(v.H_pure, v.D_weighted)[0]:+.3f}')


[4] H_pure — total_freq 잔차화 버전
  R^2(total_freq -> H_norm) = 0.679
  H_pure ~ total_freq r = -0.000  (설계상 0)
  H_pure ~ H_norm     r = +0.567
  H_pure ~ D_weighted r = +0.401


## 5. NNI의 구성 타당도(construct validity) 검증

NNI 계산에 **전혀 쓰지 않은** 변수들과의 상관을 확인해, 지수가 이론이 예측하는 방향으로
다른 지표와 연결되는지 검증한다.

In [8]:
# =========================================================
# 5. 구성 타당도 검증
# =========================================================
df['trust_curated_avg'] = df[['Q88_1', 'Q88_2', 'Q88_3']].mean(axis=1)
df['q97_1_yes'] = (df['Q97_1'] == 1).astype(int)


validity_checks = {
    'Q87_1 (뉴스 전반 신뢰도)':            ('Q87_1', '음(-) 또는 무관'),
    'Q88 평균 (알고리즘/지인 추천 신뢰도)': ('trust_curated_avg', '양(+)'),
    'Q97_1 (유료 이용 의향, Yes=1)':       ('q97_1_yes', '음(-)'),
    'DQ3 (연령)':                        ('DQ3', '음(-)'),
}

rows = []
nni_valid = df['NNI'].notna()
for label, (col, expected) in validity_checks.items():
    sub = df.loc[nni_valid & df[col].notna(), ['NNI', col]]
    r, p = pearsonr(sub['NNI'], sub[col])
    rows.append({'검증 변수': label, 'n': len(sub), '상관계수 r': round(r, 3),
                 'p-value': round(p, 4), '기대 방향': expected})

validity_table = pd.DataFrame(rows)
print(validity_table.to_string(index=False))
print()
print('해석: 대부분 기대 방향과 일치 -> NNI가 맥락 없는 소비/신뢰 저하/낮은 지불 유인을 실제로')
print('(무엇이 "언론사 인지를 흐리는 소비"인지)에 대해서는 재검토 여지가 있는 한계로 남겨둔다.')

                  검증 변수    n  상관계수 r  p-value      기대 방향
      Q87_1 (뉴스 전반 신뢰도) 5702  -0.017   0.2027 음(-) 또는 무관
Q88 평균 (알고리즘/지인 추천 신뢰도) 5702   0.213   0.0000       양(+)
Q97_1 (유료 이용 의향, Yes=1) 5702   0.131   0.0000       음(-)
               DQ3 (연령) 5702  -0.445   0.0000       음(-)

해석: 대부분 기대 방향과 일치 -> NNI가 맥락 없는 소비/신뢰 저하/낮은 지불 유인을 실제로
(무엇이 "언론사 인지를 흐리는 소비"인지)에 대해서는 재검토 여지가 있는 한계로 남겨둔다.


## 6. 인구통계 (군집화 입력이 아니라 프로파일링 전용)

나이만으로는 부족하고 성별·학력까지 함께 보는 것을 권장한다. 군집화 피처에는 넣지 않는데,
넣으면 "미디어 이용 패턴"이 아니라 "인구통계 분포"로 군집이 갈릴 위험이 있기 때문이다
(예: 연령별로만 나뉘어 버릴 수 있음). 대신 군집이 정해진 뒤 사후적으로 설명하는 데 사용한다.

In [9]:
# =========================================================
# 6. 인구통계 파생 변수 (프로파일링 전용)
# =========================================================
df['age_group'] = pd.cut(df['DQ3'], bins=[18, 29, 39, 49, 59, 200],
                          labels=['20대 이하', '30대', '40대', '50대', '60대 이상'], include_lowest=True)
df['gender'] = df['DQ2'].map({1: '남성', 2: '여성'})
df['education'] = df['BQ3'].map({1: '중졸 이하', 2: '고졸', 3: '대재 이상', 4: '원재 이상'})
print(df[['age_group', 'gender', 'education']].describe(include='all'))

       age_group gender education
count       6000   6000      6000
unique         5      2         4
top       60대 이상     여성     대재 이상
freq        2076   3030      2921


## 저장

다음 단계(03_군집분석.ipynb)에서 이어받을 수 있도록 중간 산출물을 저장한다.

In [10]:
# =========================================================
# 중간 저장: 03_군집분석.ipynb 에서 사용
# =========================================================
with open('step2_nni.pkl', 'wb') as f:
    pickle.dump({
        'df': df,
        'channel_specs': channel_specs,
        'channel_cols': channel_cols,
        'channel_tiers': channel_tiers,
        'FREQ_WINDOW': FREQ_WINDOW,
        'H_max': H_max,
        'alpha': alpha,
        'beta': beta,
    }, f)
print('저장 완료: step2_nni.pkl')

저장 완료: step2_nni.pkl